# MCP Toolbox + OpenRouter — simple cross-database tool calls

This notebook runs two tutorial questions through **OpenRouter** while the model may call **MCP Toolbox** tools (Postgres + Mongo via the **combined** toolset). The loop lives in `run_chat_with_tools` in `mcp_tutorial.agent`.

**Prerequisites:** Docker Compose up (Postgres, Mongo, **seed**, **toolbox**); from the **repo root**, a venv with **`pip install -e .`** (see `pyproject.toml`); **`OPENROUTER_API_KEY`** in the environment or repo-root **`.env`**.

**Steps**

1. **Environment** — Stack running; Toolbox reachable (default `http://127.0.0.1:5050` if you use the compose port mapping).
2. **Setup cell (below)** — Imports, logging, load `.env`, build `Settings`, import prompts (`mcp_tutorial.prompts`, same strings as `scripts/run_agent_demo.py`).
3. **Prompt A cell** — One chat with tools: US 2016 athletics/running medalists, structured data then bios.
4. **Prompt B cell** — Another chat: biography search first (randomized keyword/index from `build_prompt_b()`), then cross-check in Postgres.

The setup cell turns on **DEBUG** for `mcp_tutorial.agent` so each model round logs a short text snippet and any tool calls.

In [ ]:
# Notebook step 2 — run once before the Prompt A / B cells (see markdown above).
# Requires: ``pip install -e .`` from the repo root so ``mcp_tutorial`` imports work.

# --- Step 2a: imports (stdlib, then third party) ---
import logging
import textwrap
from pathlib import Path

from dotenv import load_dotenv

# --- Step 2b: logging ---
# DEBUG: each model round from ``mcp_tutorial.agent`` (reply snippet + tool names).
# INFO: when each ``chat.completions`` and each Toolbox tool starts/ends (helps if a call hangs).
# Third-party HTTP stacks stay at WARNING so the notebook stays readable.
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-5s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
for _name in ("httpx", "httpcore", "openai", "asyncio"):
    logging.getLogger(_name).setLevel(logging.WARNING)


def _repo_root() -> Path:
    """Walk parents until ``pyproject.toml`` is found (cwd may be ``notebooks/``)."""
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(
        "Could not find repo root (pyproject.toml). Run from inside the clone."
    )


# --- Step 2c: repo root + ``.env`` ---
ROOT = _repo_root()
load_dotenv(ROOT / ".env")

# --- Step 2d: package imports (tutorial agent, prompts, settings) ---
from mcp_tutorial.agent import run_chat_with_tools
from mcp_tutorial.prompts import PROMPT_A, TUTORIAL_SYSTEM_PROMPT as SYS, build_prompt_b
from mcp_tutorial.settings import Settings

# --- Step 2e: settings from the environment (same vars as ``scripts/run_agent_demo.py``) ---
settings = Settings.from_env()
if not settings.openrouter_api_key:
    raise RuntimeError("Set OPENROUTER_API_KEY (e.g. in .env) to run tool-calling cells.")

# Print the system prompt
print("\nSystem Prompt\n" + "-" * 40)
print(textwrap.fill(SYS, width=80))

In [ ]:
# Notebook step 3 — Prompt A (``mcp_tutorial.prompts.PROMPT_A``)

print("Prompt A\n" + "-" * 40)
print(textwrap.fill(PROMPT_A, width=80))
print("\n" + "="*60 + "\n")

out_a = run_chat_with_tools(settings, user_prompt=PROMPT_A, system_prompt=SYS)
print(out_a)

In [ ]:
# Notebook step 4 — Prompt B (``build_prompt_b()``: random theme + hit index nudge)

prompt_b, keyword_b, pick_b = build_prompt_b()
print("Prompt B\n" + "-" * 40)
print(f"(Keyword: {keyword_b!r}, pick_index: {pick_b})")
print(textwrap.fill(prompt_b, width=80))
print("\n" + "="*60 + "\n")

out_b = run_chat_with_tools(settings, user_prompt=prompt_b, system_prompt=SYS)
print(out_b)